# SigLIP / SigLIP 2 Backbone Verification

**Model**: `google/siglip-so400m-patch14-384` (SigLIP v1) or `google/siglip2-so400m-patch16-384` (SigLIP v2)  
**Arch**: ViT-SO400M (~400M params image encoder) + text encoder  
**Embed dim**: 1152 (so400m) · 768 (base/large variants)  
**Input**: 384×384 px (so400m-384), 224/256/512 for other variants  
**Access**: HuggingFace, no auth  
**Zero-shot**: Yes — paired text encoder (sigmoid loss CLIP)

SigLIP uses **sigmoid loss** instead of softmax cross-entropy for contrastive training,
enabling larger batch sizes and better small-batch performance.

Papers:  
- [SigLIP v1: Sigmoid Loss for Language Image Pre-Training (arXiv 2303.15343)](https://arxiv.org/abs/2303.15343)  
- [SigLIP 2: Better Multilingual VL Encoder (2025)](https://huggingface.co/blog/siglip2)

Available variants (both versions share the same transformers API):

| Hub ID | Res | Embed dim | Notes |
|---|---|---|---|
| `google/siglip-so400m-patch14-224` | 224 | 1152 | SigLIP v1, faster |
| `google/siglip-so400m-patch14-384` | 384 | 1152 | SigLIP v1, stronger |
| `google/siglip2-so400m-patch14-224` | 224 | 1152 | SigLIP v2 |
| `google/siglip2-so400m-patch16-384` | 384 | 1152 | SigLIP v2, recommended |
| `google/siglip2-so400m-patch16-512` | 512 | 1152 | SigLIP v2, hi-res |

In [ ]:
# %pip install -q transformers torch torchvision Pillow

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor

# Use SigLIP v2 so400m — best general-purpose variant
HUB = "google/siglip2-so400m-patch16-384"   # change to v1 or other res as needed

processor = AutoProcessor.from_pretrained(HUB)
model     = AutoModel.from_pretrained(HUB)
model.eval()

print("Hub:", HUB)
print("Vision hidden size:", model.config.vision_config.hidden_size)    # 1152 so400m
print("Projection dim:    ", model.config.vision_config.hidden_size)    # same (no separate projection)
print("Image size:        ", model.config.vision_config.image_size)     # 384

## 2. Preprocess image

In [ ]:
from PIL import Image
import numpy as np

def preprocess(path: str) -> dict:
    img = Image.open(path).convert("RGB")
    return processor(images=img, return_tensors="pt")

# Inspect processor settings
print("Crop size:  ", processor.image_processor.crop_size)
print("Image mean: ", processor.image_processor.image_mean)
print("Image std:  ", processor.image_processor.image_std)

dummy_pil = Image.fromarray(np.ones((512, 512, 3), dtype=np.uint8) * 128)
inputs    = processor(images=dummy_pil, return_tensors="pt")
print("pixel_values:", inputs["pixel_values"].shape)   # [1, 3, 384, 384]

## 3. Extract image embeddings

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    # Option A: convenience method. In transformers 5.x this returns a
    # BaseModelOutputWithPooling object; pull .pooler_output to get the tensor.
    img_feats = model.get_image_features(
        pixel_values=inputs["pixel_values"]
    ).pooler_output   # [B, 1152]

    # Option B: vision tower only (patches + pooled). SigLIP has no CLS token,
    # so last_hidden_state is already pure patch tokens — do NOT slice [:, 1:].
    vis_out = model.vision_model(pixel_values=inputs["pixel_values"])
    pooled  = vis_out.pooler_output           # [B, 1152] — attention-pool head
    patches = vis_out.last_hidden_state       # [B, N_patches, 1152]

# L2-normalize for cosine similarity
img_norm = F.normalize(img_feats, dim=-1)

print("get_image_features: ", img_feats.shape)   # [1, 1152]
print("pooler_output:      ", pooled.shape)       # [1, 1152]
# so400m-patch14-384: grid=27, N_patches=729. so400m-patch16-384: grid=24, N=576.
print("patch tokens:       ", patches.shape)
print("A == B:", torch.allclose(img_feats, pooled, atol=1e-4))


## Segmentation mode — spatial feature map

Reshape the patch sequence from section 3 to a 2-D spatial grid.
For **SigLIP so400m-384** with 16-px patches: **24×24** grid (384/16=24, 576 patches),
1152-dim. For so400m-512: 32×32 (1024 patches).

In [ ]:
import math

# patches from section 3: vis_out.last_hidden_state[:, 1:]
# SigLIP so400m-384 with 16px-patches: 384/16=24, 24×24=576 patches, D=1152
B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 24 for 384px / 16px-patches (so400m-384)
assert H * W == N, f"Patch grid {N} is not a perfect square"

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 576, 1152]
print(f"spatial map:   {spatial.shape}")   # [1, 1152, 24, 24]

## 4. Zero-shot: text embeddings & similarity

SigLIP uses **softplus**(logit_scale) with a **logit_bias** term rather than a temperature.
For zero-shot inference, use cosine similarity directly on L2-normalized embeddings.

In [ ]:
prompts = [
    "A chest X-ray showing no acute cardiopulmonary findings.",
    "Cardiomegaly is present on this chest X-ray.",
    "Bilateral pleural effusions.",
    "Pneumonia with right lower lobe consolidation.",
    "Pneumothorax visible on the left.",
]

text_inputs = processor(text=prompts, return_tensors="pt",
                        padding="max_length", truncation=True)

with torch.no_grad():
    # transformers 5.x: get_text_features returns BaseModelOutputWithPooling;
    # pull .pooler_output for the [B, D] tensor.
    text_feats = model.get_text_features(**text_inputs).pooler_output   # [5, 1152]
    text_norm  = F.normalize(text_feats, dim=-1)

sim = (img_norm @ text_norm.T).squeeze(0)   # [5]
for p, s in sorted(zip(prompts, sim.tolist()), key=lambda x: -x[1]):
    print(f"{s:.4f}  {p}")


## 5. SigLIP v1 vs v2 comparison

In [ ]:
v1_hub = "google/siglip-so400m-patch14-384"
v2_hub = "google/siglip2-so400m-patch16-384"

for hub in [v1_hub, v2_hub]:
    proc = AutoProcessor.from_pretrained(hub)
    mdl  = AutoModel.from_pretrained(hub).eval()
    inp  = proc(images=dummy_pil, return_tensors="pt")
    with torch.no_grad():
        feats = mdl.get_image_features(**inp).pooler_output   # [1, 1152]
    print(f"{hub.split('/')[-1]}: embed={feats.shape}  "
          f"input={inp['pixel_values'].shape[-2:]}")


## Notes for RadHarmony integration

```
model_call : lambda m, x: F.normalize(m.get_image_features(pixel_values=x).pooler_output, dim=-1)
pool       : None
embed_dim  : 1152 (so400m) · 768 (base/large variants)
transform  : AutoProcessor - resize/crop/normalize automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
```

**Transformers 5.x return type**: `SiglipModel.get_image_features()` returns a
`BaseModelOutputWithPooling` (not a tensor); extract `.pooler_output` to get
the `[B, D]` embedding. The vision tower's `pooler_output` is identical.

**SigLIP v1 vs v2**: SigLIP 2 uses a multi-positive contrastive objective and
multilingual training - stronger on classification and retrieval. Prefer v2 unless
you specifically need backward compatibility.

**SigLIP vs CLIP normalization**: SigLIP was trained with `mean=(0.5,0.5,0.5)`,
`std=(0.5,0.5,0.5)` - different from CLIP's `(0.48,0.46,0.41)` / `(0.27,0.26,0.28)`.
Always use the model's own `AutoProcessor`; don't borrow normalization from another backbone.

**MedSigLIP** (`google/medsiglip-448`) is SigLIP fine-tuned on medical image-text pairs
and is already in RadHarmony. This notebook is for the general-domain SigLIP as a baseline.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- SigLIP has **no CLS token**, so `last_hidden_state` is already pure patch tokens.
  Do NOT slice `[:, 1:]` — that drops a real patch.
  ```python
  patches = model.vision_model(pixel_values=x).last_hidden_state   # [B, N, D]
  ```
- Patch grid is variant-dependent (image_size / patch_size):
  - `so400m-patch14-384` -> 27 x 27 = 729 patches, `[B, 1152, 27, 27]`
  - `so400m-patch16-384` -> 24 x 24 = 576 patches, `[B, 1152, 24, 24]`
- Reshape: `patches.permute(0, 2, 1).reshape(B, D, H, W)`.
- Read the grid dynamically: `H = W = model.config.vision_config.image_size // model.config.vision_config.patch_size`.


In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

img = Image.open(CXR_PATH).convert("RGB")
inp = processor(images=img, return_tensors="pt")

with torch.no_grad():
    feats = F.normalize(model.get_image_features(**inp).pooler_output, dim=-1)   # [1, 1152]

# Zero-shot scores
sim = (feats @ text_norm.T).squeeze(0)
for p, s in sorted(zip(prompts, sim.tolist()), key=lambda x: -x[1]):
    print(f"{s:.4f}  {p}")


## Notes for RadHarmony integration

```
model_call : lambda m, x: F.normalize(m.get_image_features(pixel_values=x).pooler_output, dim=-1)
pool       : None
embed_dim  : 1152 (so400m) · 768 (base/large variants)
transform  : AutoProcessor - resize/crop/normalize automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
```

**Transformers 5.x return type**: `SiglipModel.get_image_features()` returns a
`BaseModelOutputWithPooling` (not a tensor); extract `.pooler_output` to get
the `[B, D]` embedding. The vision tower's `pooler_output` is identical.

**SigLIP v1 vs v2**: SigLIP 2 uses a multi-positive contrastive objective and
multilingual training - stronger on classification and retrieval. Prefer v2 unless
you specifically need backward compatibility.

**SigLIP vs CLIP normalization**: SigLIP was trained with `mean=(0.5,0.5,0.5)`,
`std=(0.5,0.5,0.5)` - different from CLIP's `(0.48,0.46,0.41)` / `(0.27,0.26,0.28)`.
Always use the model's own `AutoProcessor`; don't borrow normalization from another backbone.

**MedSigLIP** (`google/medsiglip-448`) is SigLIP fine-tuned on medical image-text pairs
and is already in RadHarmony. This notebook is for the general-domain SigLIP as a baseline.
